# CardiAnatomy: end-to-end BiV volumetric meshing in Google Colab

**Launch this notebook from GitHub in Colab, then run cells from top to bottom.** It connects the merged CardiAnatomy staged runner to the upstream BiV Volumetric Meshing implementation.

This notebook can download the four pretrained segmentation-network packages from [Zenodo record 15649643](https://zenodo.org/records/15649643) or the upstream-compatible packages from the [upstream v1-public release](https://github.com/cdttk/biv-volumetric-meshing/releases/tag/v1-public). It can also download openCARP's upstream-documented AppImage package after you explicitly review and acknowledge the applicable license. **UK Biobank participant-level scans are access-controlled**: for the input scans, use your approved UK Biobank workflow or a direct archive URL that you are authorized to download and transfer to Colab/Drive. The notebook does not bypass access controls.

You provide four 4D NIfTI views, or optionally a ZIP archive containing them. The notebook does not choose a dataset for you.

**Outputs and downloaded model archives are saved to your Google Drive.** Use private Drive folders and a de-identified case identifier. Never store patient identifiers, access tokens, or direct links containing credentials in the notebook or a public repository.

Important limits:
- The upstream maintainers warn that this workflow may not work as expected with scans outside its intended UK Biobank setting. Review outputs against the source scans.
- Segmentation model assets and input datasets have separate access/use terms. Verify those before use.
- Volumetric and UVC/fiber stages rely on openCARP-dependent tools; the notebook pauses for explicit license review.
- Passing mesh QC is a geometric screen, not proof of anatomical accuracy or clinical validity.

## 1. Check the runtime

For segmentation, choose Runtime → Change runtime type → T4 GPU (or another compatible GPU) in Colab. The notebook uses a separate Python 3.11 environment when the active Colab kernel is Python 3.12 or newer because the upstream README warns that nnU-Net may not work with Python 3.12+.

In [ ]:
import os, sys, platform, subprocess, shutil, json, time, hashlib, tarfile, urllib.request
from pathlib import Path
from datetime import datetime, timezone

print("Kernel Python:", sys.version)
print("Platform:", platform.platform())
print("Git:", shutil.which("git"))

## 2. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab to mount Google Drive.") from exc

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
assert DRIVE_ROOT.is_dir(), "Google Drive did not mount at /content/drive/MyDrive"
print("Drive mounted. Keep input and output folders private.")

## 3. Choose your inputs, downloads, and run settings

Edit this cell before continuing. Downloading the four network archives from Zenodo is enabled by default and requires approximately 6 GB of Drive space. You can choose the upstream release packages instead, or point to model archives/workspace already stored in Drive.

For UK Biobank scans, **approved access and transfer conditions still apply**. UK Biobank provides data through its authorized researcher workflow, using the UK Biobank Research Analysis Platform by default; this notebook cannot grant access. If your approved data workflow permits a ZIP of the four views to be transferred to Colab, enter that direct HTTPS URL below. Otherwise, put the permitted files in your private Drive folder and set the four paths.

Links: [UK Biobank access](https://www.ukbiobank.ac.uk/use-our-data/apply-for-access/), [Zenodo model record](https://zenodo.org/records/15649643), [upstream model release](https://github.com/cdttk/biv-volumetric-meshing/releases/tag/v1-public), [openCARP installation](https://opencarp.org/download/installation), [openCARP license](https://opencarp.org/download/license).

The default stage list requests segmentation, contours, surfaces, and volumetric meshing. Set STAGES to [`"segmentation", "contour", "surface"`] for the surface stages only. Add `"uvc-fiber"` after `"volumetric"` only after checking its prerequisites and license.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# OPTION A: inputs already in your private Drive folder
SAX_PATH     = DRIVE_ROOT / "EDIT_ME" / "case" / "SAX.nii.gz"
LAX_2CH_PATH = DRIVE_ROOT / "EDIT_ME" / "case" / "LAX_2Ch.nii.gz"
LAX_3CH_PATH = DRIVE_ROOT / "EDIT_ME" / "case" / "LAX_3Ch.nii.gz"
LAX_4CH_PATH = DRIVE_ROOT / "EDIT_ME" / "case" / "LAX_4Ch.nii.gz"

# OPTION B: set a direct HTTPS URL to a ZIP of those four files, but only if your data
# agreement/source explicitly permits downloading/transferring this archive to Colab/Drive.
DOWNLOAD_AUTHORIZED_INPUT_ARCHIVE = False
UKB_INPUT_ARCHIVE_URL = ""  # Leave blank to enter the signed URL privately when prompted below.
UKB_INPUT_ARCHIVE_SHA256 = ""  # optional but strongly recommended if your archive provider supplies it
UKB_DATA_ACCESS_APPROVED = False
UKB_EXPORT_TO_COLAB_ALLOWED = False
UKB_USE_BEARER_TOKEN = False  # for an endpoint that explicitly documents bearer-token HTTP downloads

# Download pretrained models into this private Drive cache by default.
DOWNLOAD_PRETRAINED_MODELS = True
MODEL_DOWNLOAD_SOURCE = "zenodo"  # "zenodo", "upstream_release", or "existing_drive_folder"
MODEL_ASSET_SOURCE = DRIVE_ROOT / "CardiAnatomy_ModelDownloads"
# If MODEL_DOWNLOAD_SOURCE="existing_drive_folder", change MODEL_ASSET_SOURCE to your existing folder.

DRIVE_RUNS_DIR = DRIVE_ROOT / "CardiAnatomy_BiV_Runs"
CASE_ID = "case_001"   # pseudonymous path-safe ID; never a patient name/MRN/DOB
INSTANCE = 2
TIMEFRAME = None
UPSTREAM_REF = "v1-public"

# Ordered stage prefix. For surface-only work use ["segmentation", "contour", "surface"].
STAGES = ["segmentation", "contour", "surface", "volumetric"]
# Optionally append "uvc-fiber" after "volumetric".

PREFER_GPU = True
STAGE_TIMEOUT_SECONDS = 4 * 60 * 60
MINIMUM_POINTS = 500
MINIMUM_TETRAHEDRA = 1000
MINIMUM_TETRA_SHAPE_QUALITY = 0.05

# Set True only after reviewing the applicable access/use conditions.
DATA_TERMS_REVIEWED = False
MODEL_ASSET_TERMS_REVIEWED = False

# Allowed: "not_reviewed", "academic_noncommercial", "commercial_product_development", "other".
INTENDED_USE = "not_reviewed"
OPENCARP_TERMS_REVIEWED = False
OPENCARP_COMMERCIAL_LICENSE_CONFIRMED = False
INSTALL_OPENCARP = True  # downloads the upstream-documented installer only after license gates pass
OPENCARP_BIN_DIR = None  # optionally use an existing separately licensed installation.

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_ROOT = DRIVE_RUNS_DIR / f"{CASE_ID}_{RUN_ID}"
MODEL_WORKSPACE = Path("/content") / f"cardianatomy_nnunet_workspace_{RUN_ID}"
print("Run ID:", RUN_ID)
print("Output folder:", OUTPUT_ROOT)
print("Model download source:", MODEL_DOWNLOAD_SOURCE if DOWNLOAD_PRETRAINED_MODELS else "existing assets")
print("Stages:", STAGES)

## 4. Acquire UK Biobank input data (only through an approved workflow)

UK Biobank participant data are access-controlled. UK Biobank says its Research Analysis Platform is the default access route, and data use is governed by eligibility, research approvals and a legal agreement. This notebook does not log in to UKB-RAP or attempt to bypass it.

Choose one:
- **Already in your private Drive:** set the four input paths in the configuration cell above.
- **Authorized direct archive:** set `UKB_INPUT_ARCHIVE_URL` to a direct HTTPS URL for a ZIP containing exactly `SAX.nii.gz`, `LAX_2Ch.nii.gz`, `LAX_3Ch.nii.gz`, and `LAX_4Ch.nii.gz`; set `UKB_DATA_ACCESS_APPROVED = True` and `UKB_EXPORT_TO_COLAB_ALLOWED = True` only if that export is permitted by your approved data workflow. A bearer token is prompted privately only if the server explicitly requires it.

Official access guidance: [UK Biobank access](https://www.ukbiobank.ac.uk/use-our-data/apply-for-access/) and [access to UK Biobank data](https://www.ukbiobank.ac.uk/about-us/how-we-work/access-to-uk-biobank-data/). If your approval requires processing to remain inside UKB-RAP, do not download the scans to Colab or Drive; run in an approved environment instead.

In [ ]:
import hashlib, time, zipfile, stat
from urllib.parse import urlparse
from urllib.request import Request, urlopen
from getpass import getpass

# Streaming download helper: supports resuming a partial .part file and optional checksum validation.
def file_digest(path, algorithm="sha256", chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_https_file(url, destination, *, expected_checksum=None, checksum_algorithm="sha256", headers=None):
    destination = Path(destination)
    if urlparse(url).scheme.lower() != "https":
        raise ValueError("Downloads must use HTTPS.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".part")
    if destination.is_file():
        if expected_checksum:
            actual = file_digest(destination, checksum_algorithm)
            if actual.lower() == expected_checksum.lower():
                print(f"Already downloaded and checksum verified: {destination.name}")
                return destination
            raise ValueError(f"Existing file checksum mismatch; refusing to overwrite {destination}")
        print(f"Using existing download: {destination.name}")
        return destination

    offset = partial.stat().st_size if partial.exists() else 0
    request_headers = dict(headers or {})
    if offset:
        request_headers["Range"] = f"bytes={offset}-"
    request = Request(url, headers=request_headers)
    with urlopen(request, timeout=120) as response:
        status = getattr(response, "status", 200)
        if offset and status == 206:
            mode, start = "ab", offset
        else:
            mode, start = "wb", 0
            offset = 0
        length_header = response.headers.get("Content-Length")
        total = start + int(length_header) if length_header and length_header.isdigit() else None
        try:
            from tqdm.auto import tqdm
            bar = tqdm(total=total, initial=start, unit="B", unit_scale=True, desc=destination.name)
        except Exception:
            bar = None
        with partial.open(mode) as stream:
            while True:
                chunk = response.read(8 * 1024 * 1024)
                if not chunk:
                    break
                stream.write(chunk)
                if bar is not None:
                    bar.update(len(chunk))
        if bar is not None:
            bar.close()

    if expected_checksum:
        actual = file_digest(partial, checksum_algorithm)
        if actual.lower() != expected_checksum.lower():
            raise ValueError(
                f"Checksum validation failed for {destination.name}: expected {expected_checksum}, got {actual}. "
                f"Partial file preserved at {partial} for inspection."
            )
    partial.replace(destination)
    print(f"Saved: {destination} ({destination.stat().st_size:,} bytes)")
    if expected_checksum:
        print(f"Verified {checksum_algorithm.upper()}: {expected_checksum}")
    return destination

input_archive_url = UKB_INPUT_ARCHIVE_URL.strip()
if DOWNLOAD_AUTHORIZED_INPUT_ARCHIVE and not input_archive_url:
    input_archive_url = getpass("Paste the authorized direct HTTPS input-archive URL (input hidden; not printed): ").strip()

if input_archive_url:
    if not UKB_DATA_ACCESS_APPROVED or not UKB_EXPORT_TO_COLAB_ALLOWED:
        raise PermissionError(
            "The input archive URL is set, but the notebook will not download it until you confirm both "
            "approved data access and permission to export/transfer the data to Colab/Drive."
        )
    if "EDIT_ME" in CASE_ID:
        raise ValueError("Set CASE_ID before downloading an input archive.")
    headers = {}
    if UKB_USE_BEARER_TOKEN:
        token = getpass("Enter the bearer token for this authorized download (input hidden; not printed): ")
        if not token:
            raise ValueError("Bearer token was blank.")
        headers["Authorization"] = f"Bearer {token}"
    input_cache = DRIVE_ROOT / "CardiAnatomy_InputDownloads" / CASE_ID / RUN_ID
    archive_path = input_cache / "authorized_four_view_inputs.zip"
    download_https_file(
        input_archive_url,
        archive_path,
        expected_checksum=UKB_INPUT_ARCHIVE_SHA256.strip() or None,
        checksum_algorithm="sha256",
        headers=headers,
    )
    extract_dir = input_cache / "extracted"
    extract_dir.mkdir(parents=True, exist_ok=False)
    expected_names = {
        "SAX.nii.gz": "SAX",
        "LAX_2Ch.nii.gz": "LAX_2CH",
        "LAX_3Ch.nii.gz": "LAX_3CH",
        "LAX_4Ch.nii.gz": "LAX_4CH",
    }
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            target = (extract_dir / info.filename).resolve()
            if not target.is_relative_to(extract_dir.resolve()):
                raise ValueError(f"Unsafe path in input ZIP: {info.filename}")
            unix_mode = info.external_attr >> 16
            if stat.S_ISLNK(unix_mode):
                raise ValueError(f"Refusing symbolic link in input ZIP: {info.filename}")
        archive.extractall(extract_dir)
    found = {}
    for filename, key in expected_names.items():
        matches = [path for path in extract_dir.rglob(filename) if path.is_file()]
        if len(matches) != 1:
            raise FileNotFoundError(f"Expected exactly one {filename} in the ZIP, found {len(matches)}.")
        found[key] = matches[0]
    SAX_PATH, LAX_2CH_PATH, LAX_3CH_PATH, LAX_4CH_PATH = (
        found["SAX"], found["LAX_2CH"], found["LAX_3CH"], found["LAX_4CH"]
    )
    print("Extracted and mapped the four input views into:", extract_dir)
elif DOWNLOAD_AUTHORIZED_INPUT_ARCHIVE:
    raise ValueError("DOWNLOAD_AUTHORIZED_INPUT_ARCHIVE=True but no archive URL was supplied.")
else:
    print("No direct input archive download requested; using the four Drive paths from the configuration cell.")

print("Input paths configured. The next cell validates their existence, geometry and hashes.")

## 5. Validate the chosen NIfTI views

This reads NIfTI headers, reports shapes, voxel spacing and orientation codes, and calculates SHA-256 hashes. It does not alter source files, resample images or perform registration.

In [ ]:
try:
    import nibabel as nib
    import numpy as np
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "nibabel", "numpy"], check=True)
    import nibabel as nib
    import numpy as np

VIEWS = {
    "SAX": Path(SAX_PATH),
    "LAX_2CH": Path(LAX_2CH_PATH),
    "LAX_3CH": Path(LAX_3CH_PATH),
    "LAX_4CH": Path(LAX_4CH_PATH),
}
if not CASE_ID or not CASE_ID.replace("_", "").replace("-", "").isalnum():
    raise ValueError("CASE_ID must be a simple path-safe alphanumeric identifier.")
if any("EDIT_ME" in path.parts for path in VIEWS.values()):
    raise ValueError("Edit the four input paths, or configure an authorized input archive download in the previous cell.")
allowed_stages = ["segmentation", "contour", "surface", "volumetric", "uvc-fiber"]
if not STAGES or STAGES != allowed_stages[:len(STAGES)]:
    raise ValueError("STAGES must be a non-empty sequential prefix: segmentation, contour, surface, volumetric, uvc-fiber.")
if not DATA_TERMS_REVIEWED:
    raise RuntimeError("Review the data access/use terms, then set DATA_TERMS_REVIEWED = True.")
if not MODEL_ASSET_TERMS_REVIEWED:
    raise RuntimeError("Review the model asset access/use terms, then set MODEL_ASSET_TERMS_REVIEWED = True.")

def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_report = {}
for view_name, path in VIEWS.items():
    if not path.is_file():
        raise FileNotFoundError(f"{view_name} file does not exist: {path}")
    img = nib.load(str(path), mmap=True)
    shape = tuple(int(v) for v in img.shape)
    if len(shape) != 4 or any(v < 1 for v in shape):
        raise ValueError(f"{view_name} must be a 4D cine NIfTI; got shape {shape}")
    affine = np.asarray(img.affine, dtype=float)
    if affine.shape != (4, 4) or not np.isfinite(affine).all() or abs(np.linalg.det(affine[:3, :3])) < 1e-12:
        raise ValueError(f"{view_name} has a non-finite or singular affine.")
    spacing = [float(v) for v in nib.affines.voxel_sizes(affine)[:3]]
    axcodes = tuple(nib.orientations.aff2axcodes(affine))
    input_report[view_name] = {
        "path": str(path), "shape": shape, "voxel_spacing_mm": spacing,
        "axis_codes": axcodes, "sha256": sha256_file(path),
    }
    print(f"{view_name}: shape={shape}; spacing_mm={[round(v, 4) for v in spacing]}; "
          f"axis_codes={axcodes}; SHA256={input_report[view_name]['sha256']}")

if len({tuple(row["axis_codes"]) for row in input_report.values()}) != 1:
    raise ValueError("The four views have different affine axis-code conventions. Resolve this explicitly; the notebook will not silently reorient them.")
print("\nHeader checks passed. This does NOT verify view registration, temporal consistency or anatomical coverage.")

## 6. Set up a compatible Python environment and clone the code

This installs dependencies from the upstream project's requirements file and installs CardiAnatomy with its I/O extra. A separate Python 3.11 environment is used when necessary. The install can download several gigabytes, particularly PyTorch and nnU-Net dependencies.

In [ ]:
import os, sys, subprocess, shutil, urllib.request, tarfile
from pathlib import Path

def run_command(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

REPO_ROOT = Path("/content/Virelion-CardiAnatomy")
UPSTREAM_ROOT = Path("/content/biv-volumetric-meshing")
if not REPO_ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "https://github.com/Virelion-Biotech/Virelion-CardiAnatomy.git", REPO_ROOT])
elif not (REPO_ROOT / ".git").exists():
    raise RuntimeError(f"{REPO_ROOT} exists but is not a git checkout. Rename it manually; files will not be overwritten.")
if not UPSTREAM_ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "--branch", UPSTREAM_REF, "https://github.com/cdttk/biv-volumetric-meshing.git", UPSTREAM_ROOT])
elif not (UPSTREAM_ROOT / ".git").exists():
    raise RuntimeError(f"{UPSTREAM_ROOT} exists but is not a git checkout. Rename it manually; files will not be overwritten.")
upstream_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip()
print("Upstream commit:", upstream_sha, "| ref:", UPSTREAM_REF)

ENV_ROOT = Path("/content/cardianatomy-py311")
if sys.version_info < (3, 12):
    if not ENV_ROOT.exists():
        run_command([sys.executable, "-m", "venv", ENV_ROOT])
    ENV_PYTHON = ENV_ROOT / "bin" / "python"
else:
    MICROMAMBA = Path("/content/micromamba")
    if not MICROMAMBA.exists():
        archive = Path("/content/micromamba.tar.bz2")
        urllib.request.urlretrieve("https://micro.mamba.pm/api/micromamba/linux-64/latest", archive)
        with tarfile.open(archive, "r:bz2") as tar:
            member = tar.getmember("bin/micromamba")
            stream = tar.extractfile(member)
            if stream is None:
                raise RuntimeError("Could not extract micromamba executable.")
            with MICROMAMBA.open("wb") as output:
                shutil.copyfileobj(stream, output)
        MICROMAMBA.chmod(0o755)
    if not (ENV_ROOT / "bin" / "python").exists():
        run_command([MICROMAMBA, "create", "-y", "-p", ENV_ROOT, "-c", "conda-forge", "python=3.11", "pip"])
    ENV_PYTHON = ENV_ROOT / "bin" / "python"

if not ENV_PYTHON.exists():
    raise FileNotFoundError(f"Python environment was not created: {ENV_PYTHON}")
print("Pipeline Python:", subprocess.check_output([ENV_PYTHON, "--version"], text=True).strip())

constraints = Path("/content/cardianatomy-constraints.txt")
constraints.write_text("numpy<2\nscipy<2\n", encoding="utf-8")
run_command([ENV_PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
run_command([ENV_PYTHON, "-m", "pip", "install", "-c", constraints, "-r", UPSTREAM_ROOT / "requirements.txt"])
run_command([ENV_PYTHON, "-m", "pip", "install", "-e", str(REPO_ROOT) + "[io]"])
verification = subprocess.check_output(
    [ENV_PYTHON, "-c", "import sys, torch, nibabel, meshio, cardianatomy; print(sys.version.split()[0]); print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('cardianatomy', cardianatomy.__file__)"],
    text=True,
)
print(verification)
CUDA_AVAILABLE = subprocess.check_output([ENV_PYTHON, "-c", "import torch; print(torch.cuda.is_available())"], text=True).strip() == "True"
USE_GPU = bool(PREFER_GPU and CUDA_AVAILABLE)
if PREFER_GPU and not CUDA_AVAILABLE:
    print("WARNING: no CUDA-enabled PyTorch runtime detected. Segmentation will run on CPU and may be very slow.")
print("Segmentation GPU enabled:", USE_GPU)

## 7. Download and prepare pretrained segmentation models

The Zenodo record provides four view-specific pretrained model ZIP files, approximately 1.5 GB each. Only those four files are downloaded—not the full 113.5 GB record of representative meshes and other assets. MD5 checks are verified against the published record metadata. Downloads are cached in Drive for reuse across sessions.

**License caution:** the Zenodo landing page does not make a clear, explicit model-weights licence visible in the record details inspected for this notebook. Confirm the terms with the depositor/source before proceeding, especially for redistribution or commercial/product-development use. A public download link is not itself a licence.

Choose `MODEL_DOWNLOAD_SOURCE = "zenodo"` for [Zenodo record 15649643](https://zenodo.org/records/15649643), or `"upstream_release"` for the four model tar archives attached to the [upstream v1-public GitHub release](https://github.com/cdttk/biv-volumetric-meshing/releases/tag/v1-public). The upstream release tar files match the upstream README's documented workspace format; the Zenodo ZIPs are normalized into the expected nnU-Net workspace layout by this cell. If the archive layout changes upstream, the cell will stop with an explicit error rather than running segmentation with incomplete models.

The notebook copies/extracts models to local Colab storage because the segmentation stage creates temporary files in its nnU-Net workspace. Your downloaded archives remain in Drive; the notebook never deletes your source model files.

In [ ]:
import shutil, tarfile, zipfile, stat, time
from pathlib import Path

MODEL_ASSET_SOURCE = Path(MODEL_ASSET_SOURCE)
if DOWNLOAD_PRETRAINED_MODELS:
    MODEL_ASSET_SOURCE.mkdir(parents=True, exist_ok=True)

ZENODO_MODELS = {
    "Dataset100_UKBB_Petersen_SAX.zip": {
        "url": "https://zenodo.org/records/15649643/files/Dataset100_UKBB_Petersen_SAX.zip?download=1",
        "checksum": "96d26a957a7b5f06b8d513e95f097b7d", "algorithm": "md5",
    },
    "Dataset101_UKBB_LAX_2Ch.zip": {
        "url": "https://zenodo.org/records/15649643/files/Dataset101_UKBB_LAX_2Ch.zip?download=1",
        "checksum": "ce2a42c763af0e94b679375f8533435c", "algorithm": "md5",
    },
    "Dataset102_UKBB_LAX_3Ch.zip": {
        "url": "https://zenodo.org/records/15649643/files/Dataset102_UKBB_LAX_3Ch.zip?download=1",
        "checksum": "f3e35815d972d889d4134580f2658d89", "algorithm": "md5",
    },
    "Dataset103_UKBB_LAX_4Ch.zip": {
        "url": "https://zenodo.org/records/15649643/files/Dataset103_UKBB_LAX_4Ch.zip?download=1",
        "checksum": "f32adbec76dbe91746af43fa4b9d4c65", "algorithm": "md5",
    },
}
UPSTREAM_RELEASE_MODELS = {
    "Dataset100_UKBB_Petersen_SAX.20240108.tar": {
        "url": "https://github.com/cdttk/biv-volumetric-meshing/releases/download/v1-public/Dataset100_UKBB_Petersen_SAX.20240108.tar",
        "checksum": "7ec54e28809ae8f3b24cd75ecbacdb5d5852ac29375d8d516d99db3b2100fd00", "algorithm": "sha256",
    },
    "Dataset101_UKBB_LAX_2Ch.20240108.tar": {
        "url": "https://github.com/cdttk/biv-volumetric-meshing/releases/download/v1-public/Dataset101_UKBB_LAX_2Ch.20240108.tar",
        "checksum": "832e221c85788380ba8b137adcb2e592b0c4940bad9d3492f0185f18a238b682", "algorithm": "sha256",
    },
    "Dataset102_UKBB_LAX_3Ch.20240108.tar": {
        "url": "https://github.com/cdttk/biv-volumetric-meshing/releases/download/v1-public/Dataset102_UKBB_LAX_3Ch.20240108.tar",
        "checksum": "7a9fc8f1d55d75821c37c8ace59f203624a5651ab73b5a163f808bd91a6ae435", "algorithm": "sha256",
    },
    "Dataset103_UKBB_LAX_4Ch.20240108.tar": {
        "url": "https://github.com/cdttk/biv-volumetric-meshing/releases/download/v1-public/Dataset103_UKBB_LAX_4Ch.20240108.tar",
        "checksum": "38e84a7c902123dd37f005957768467f5ef0c17b3c61d3c22c7970f7275cbc21", "algorithm": "sha256",
    },
}

if DOWNLOAD_PRETRAINED_MODELS:
    if not MODEL_ASSET_TERMS_REVIEWED:
        raise RuntimeError("Review the model-asset terms and record access/use authorization in the configuration cell before downloading.")
    if MODEL_DOWNLOAD_SOURCE == "zenodo":
        model_specs = ZENODO_MODELS
    elif MODEL_DOWNLOAD_SOURCE == "upstream_release":
        model_specs = UPSTREAM_RELEASE_MODELS
    else:
        raise ValueError("MODEL_DOWNLOAD_SOURCE must be 'zenodo' or 'upstream_release' when DOWNLOAD_PRETRAINED_MODELS=True.")
    print(f"Downloading/verifying {len(model_specs)} model archives from {MODEL_DOWNLOAD_SOURCE!r}. These are large files (~6 GB total).")
    for filename, spec in model_specs.items():
        download_https_file(
            spec["url"], MODEL_ASSET_SOURCE / filename,
            expected_checksum=spec["checksum"], checksum_algorithm=spec["algorithm"],
        )
elif not MODEL_ASSET_SOURCE.is_dir():
    raise FileNotFoundError(f"Existing model asset folder not found: {MODEL_ASSET_SOURCE}")

required_prefixes = [
    "Dataset100_UKBB_Petersen_SAX",
    "Dataset101_UKBB_LAX_2Ch",
    "Dataset102_UKBB_LAX_3Ch",
    "Dataset103_UKBB_LAX_4Ch",
]
if MODEL_WORKSPACE.exists():
    if not (MODEL_WORKSPACE / ".cardianatomy_managed_workspace").exists():
        raise RuntimeError(f"Workspace exists but is not marked as managed by this notebook: {MODEL_WORKSPACE}. Use a fresh RUN_ID; it will not be deleted.")
else:
    MODEL_WORKSPACE.mkdir(parents=True)

    if (MODEL_ASSET_SOURCE / "nnUNet_raw").is_dir():
        print("Copying extracted workspace from Drive to local Colab storage...")
        shutil.copytree(MODEL_ASSET_SOURCE, MODEL_WORKSPACE, dirs_exist_ok=True)
    else:
        archives = []
        for path in MODEL_ASSET_SOURCE.iterdir():
            if path.is_file() and (path.name.endswith(".zip") or path.name.endswith(".tar") or path.name.endswith(".tar.gz")):
                archives.append(path)
        if not archives:
            raise FileNotFoundError("No model ZIP/TAR archives found in MODEL_ASSET_SOURCE; select a download source or provide archives.")

        staging_root = Path("/content") / f"cardianatomy_model_extract_{RUN_ID}"
        staging_root.mkdir(parents=True, exist_ok=False)

        def safe_extract_archive(archive_path, destination):
            destination = Path(destination).resolve()
            if zipfile.is_zipfile(archive_path):
                with zipfile.ZipFile(archive_path) as archive:
                    for info in archive.infolist():
                        target = (destination / info.filename).resolve()
                        if not target.is_relative_to(destination):
                            raise ValueError(f"Unsafe path in ZIP {archive_path.name}: {info.filename}")
                        mode = info.external_attr >> 16
                        if stat.S_ISLNK(mode):
                            raise ValueError(f"Refusing symlink in ZIP {archive_path.name}: {info.filename}")
                    archive.extractall(destination)
            else:
                with tarfile.open(archive_path, "r:*") as archive:
                    for member in archive.getmembers():
                        target = (destination / member.name).resolve()
                        if not target.is_relative_to(destination):
                            raise ValueError(f"Unsafe path in TAR {archive_path.name}: {member.name}")
                        if member.issym() or member.islnk():
                            raise ValueError(f"Refusing link member in TAR {archive_path.name}: {member.name}")
                    archive.extractall(destination)

        def merge_dir_contents(source_dir, target_dir):
            target_dir.mkdir(parents=True, exist_ok=True)
            for child in source_dir.iterdir():
                target = target_dir / child.name
                if child.is_dir():
                    shutil.copytree(child, target, dirs_exist_ok=True)
                else:
                    shutil.copy2(child, target)

        for archive_path in archives:
            extract_root = staging_root / (archive_path.name.replace(".", "_"))
            extract_root.mkdir()
            print(f"Extracting {archive_path.name} (large archive)...")
            safe_extract_archive(archive_path, extract_root)
            has_standard_roots = False
            for dirname in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
                for source_dir in list(extract_root.rglob(dirname)):
                    if source_dir.is_dir():
                        has_standard_roots = True
                        merge_dir_contents(source_dir, MODEL_WORKSPACE / dirname)
            # Zenodo bundles can put a Dataset folder at archive root instead of
            # inside nnUNet_raw/nnUNet_results, or place trainer files at ZIP root.
            dataset_dirs_found = False
            expected_prefix = next((p for p in required_prefixes if archive_path.name.startswith(p)), None)
            for dataset_dir in list(extract_root.rglob("Dataset*")):
                if not dataset_dir.is_dir() or not any(dataset_dir.name.startswith(p) for p in required_prefixes):
                    continue
                dataset_dirs_found = True
                relative_parts = dataset_dir.relative_to(extract_root).parts
                if any(part in ("nnUNet_raw", "nnUNet_results", "nnUNet_preprocessed") for part in relative_parts[:-1]):
                    continue
                has_checkpoint = any(p.suffix == ".pth" for p in dataset_dir.rglob("*") if p.is_file())
                target_name = "nnUNet_results" if has_checkpoint else "nnUNet_raw"
                shutil.copytree(dataset_dir, MODEL_WORKSPACE / target_name / dataset_dir.name, dirs_exist_ok=True)
            # Fallback for a model ZIP whose root consists of trainer/fold files but
            # has no enclosing Dataset* directory. Attribute it only to the matching
            # view archive and require actual checkpoint files later.
            if expected_prefix and not dataset_dirs_found and not has_standard_roots:
                checkpoint_files = [p for p in extract_root.rglob("*.pth") if p.is_file()]
                if checkpoint_files:
                    result_dir = MODEL_WORKSPACE / "nnUNet_results" / expected_prefix
                    result_dir.mkdir(parents=True, exist_ok=True)
                    merge_dir_contents(extract_root, result_dir)

        # If the model archives contain the result folders but no raw dataset folder,
        # create a lightweight ID/metadata directory because the upstream predictor scans nnUNet_raw
        # to discover the model dataset name. Model weights must still exist under nnUNet_results.
        for prefix in required_prefixes:
            result_root = MODEL_WORKSPACE / "nnUNet_results"
            result_matches = [p for p in result_root.iterdir() if p.is_dir() and p.name.startswith(prefix)] if result_root.is_dir() else []
            if result_matches and not any(
                p.is_dir() and p.name.startswith(prefix)
                for p in (MODEL_WORKSPACE / "nnUNet_raw").iterdir()
            ) if (MODEL_WORKSPACE / "nnUNet_raw").is_dir() else bool(result_matches):
                result_dir = result_matches[0]
                raw_dir = MODEL_WORKSPACE / "nnUNet_raw" / result_dir.name
                raw_dir.mkdir(parents=True, exist_ok=True)
                dataset_json = next(result_dir.rglob("dataset.json"), None)
                if dataset_json:
                    shutil.copy2(dataset_json, raw_dir / "dataset.json")

        shutil.rmtree(staging_root)
    (MODEL_WORKSPACE / ".cardianatomy_managed_workspace").write_text(
        f"Created by Colab notebook on {RUN_ID}\nSource: {MODEL_ASSET_SOURCE}\n",
        encoding="utf-8",
    )

nnunet_raw = MODEL_WORKSPACE / "nnUNet_raw"
nnunet_results = MODEL_WORKSPACE / "nnUNet_results"
if not nnunet_raw.is_dir():
    raise FileNotFoundError(
        f"No nnUNet_raw directory found after model preparation: {nnunet_raw}. "
        "Try MODEL_DOWNLOAD_SOURCE='upstream_release' or inspect the upstream archive layout."
    )
if not nnunet_results.is_dir():
    raise FileNotFoundError(
        f"No nnUNet_results directory found: {nnunet_results}. "
        "The downloaded archive must include pretrained checkpoint files; a dataset metadata folder alone is not sufficient."
    )

for prefix in required_prefixes:
    raw_matches = [p for p in nnunet_raw.iterdir() if p.is_dir() and p.name.startswith(prefix)]
    result_matches = [p for p in nnunet_results.iterdir() if p.is_dir() and p.name.startswith(prefix)]
    if len(raw_matches) != 1:
        raise RuntimeError(f"Expected exactly one nnUNet_raw directory beginning {prefix!r}; found {[p.name for p in raw_matches]}")
    if len(result_matches) != 1:
        raise RuntimeError(f"Expected exactly one nnUNet_results directory beginning {prefix!r}; found {[p.name for p in result_matches]}")
    checkpoints = [p for p in result_matches[0].rglob("*.pth") if p.is_file()]
    if not checkpoints:
        raise FileNotFoundError(f"No .pth checkpoint found for {prefix} in {result_matches[0]}")
    print(f"{prefix}: raw metadata OK; checkpoints={len(checkpoints)}")
(MODEL_WORKSPACE / ".cardianatomy_managed_workspace").write_text(
    f"Validated workspace on {RUN_ID}\n",
    encoding="utf-8",
)
print("nnU-Net workspace ready:", MODEL_WORKSPACE)
print("Model archives are cached in Drive and source files remain unchanged.")

## 8. Download/install openCARP only after the license review

The upstream BiV README documents an openCARP v11.0 AppImage archive, which this notebook can download and extract automatically when `INSTALL_OPENCARP = True`. This version is retained here as the version documented by the BiV pipeline; it is not an assertion that every newer openCARP version has been tested with this adapter.

Current openCARP licensing distinguishes Apache-2.0 components from simulator binaries subject to the non-commercial Academic Public License. Commercial use requires a commercial license. Review the [official license](https://opencarp.org/download/license) and [official installation guide](https://opencarp.org/download/installation) first. The installer is deliberately restricted to a declared academic non-commercial run; for commercial/product development, configure a separately licensed installation and confirm that license. The notebook does not grant rights or bypass license terms.

In [ ]:
needs_opencarp = any(stage in STAGES for stage in ("volumetric", "uvc-fiber"))
if needs_opencarp:
    if not OPENCARP_TERMS_REVIEWED:
        raise RuntimeError("Volumetric/UVC-fiber is selected. Review https://opencarp.org/download/license and set OPENCARP_TERMS_REVIEWED = True only after review.")
    if INTENDED_USE == "academic_noncommercial":
        pass
    elif INTENDED_USE == "commercial_product_development":
        if not OPENCARP_COMMERCIAL_LICENSE_CONFIRMED:
            raise RuntimeError("Commercial/product-development use selected, but no commercial openCARP license is confirmed. Obtain a license or remove volumetric/UVC-fiber from STAGES.")
    else:
        raise RuntimeError("Set INTENDED_USE to the actual intended use before running openCARP-dependent stages.")
    if INSTALL_OPENCARP:
        if INTENDED_USE != "academic_noncommercial":
            raise RuntimeError("The optional AppImage installer is limited to reviewed academic non-commercial use. For commercial use, configure a separately licensed installation.")
        if not Path("/content/openCARP/usr/bin").is_dir():
            url = "https://git.opencarp.org/api/v4/projects/16/packages/generic/opencarp-appimage/v11.0/openCARP-v11.0-x86_64_AppImage.tar.gz"
            opencarp_cache = DRIVE_ROOT / "CardiAnatomy_ModelDownloads" / "openCARP"
            opencarp_cache.mkdir(parents=True, exist_ok=True)
            package_archive = opencarp_cache / "openCARP-v11.0-x86_64_AppImage.tar.gz"
            print("Downloading/caching upstream-documented openCARP v11.0 AppImage package in Drive...")
            download_https_file(url, package_archive)
            package_dir = Path("/content") / f"openCARP-appimage-package-{RUN_ID}"
            package_dir.mkdir(parents=True, exist_ok=True)
            package_dir.mkdir(parents=True, exist_ok=True)
            subprocess.run(["tar", "-xzf", str(package_archive), "-C", str(package_dir)], check=True)
            appimages = list(package_dir.rglob("*.AppImage"))
            if not appimages:
                raise FileNotFoundError("The openCARP archive contained no .AppImage file.")
            appimage = appimages[0]
            appimage.chmod(0o755)
            subprocess.run([str(appimage), "--appimage-extract"], cwd=package_dir, check=True)
            extracted = package_dir / "squashfs-root"
            if not (extracted / "usr" / "bin").is_dir():
                raise FileNotFoundError("Expected usr/bin in extracted openCARP AppImage.")
            shutil.move(str(extracted), "/content/openCARP")
        OPENCARP_BIN_DIR = "/content/openCARP/usr/bin"
    if not OPENCARP_BIN_DIR or not Path(OPENCARP_BIN_DIR).is_dir():
        raise FileNotFoundError("Set OPENCARP_BIN_DIR to an existing openCARP binary directory, or set INSTALL_OPENCARP = True for the reviewed academic non-commercial installation.")
    print("openCARP binaries:", OPENCARP_BIN_DIR)
    print("License review recorded:", OPENCARP_TERMS_REVIEWED, "| Intended use:", INTENDED_USE)
else:
    print("No openCARP-dependent stage selected.")

## 9. Run the selected stages

The runner writes manifests and stage logs under OUTPUT_ROOT on Drive. It rejects an existing non-empty output directory rather than overwriting results. If a run fails, preserve its folder and inspect the manifest and logs before starting a fresh run.

The progress poll reads the run manifest while the pipeline is running. The adapter writes stage records and partial artifacts as it proceeds. Keep the Colab session connected until the process completes.

In [ ]:
from pathlib import Path
import os, json, time, subprocess, shlex, shutil

DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
if OUTPUT_ROOT.exists():
    raise FileExistsError(f"Output directory already exists; results will not be overwritten: {OUTPUT_ROOT}. Rerun the configuration cell to generate a new RUN_ID.")

cmd = [
    str(ENV_PYTHON), "-m", "cardianatomy.biv_volumetric_cli",
    "--sax", str(VIEWS["SAX"]),
    "--lax-2ch", str(VIEWS["LAX_2CH"]),
    "--lax-3ch", str(VIEWS["LAX_3CH"]),
    "--lax-4ch", str(VIEWS["LAX_4CH"]),
    "--upstream-root", str(UPSTREAM_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--subject-id", CASE_ID,
    "--instance", str(INSTANCE),
    "--nnunet-workspace", str(MODEL_WORKSPACE),
    "--stages", *STAGES,
    "--python-executable", str(ENV_PYTHON),
    "--timeout-seconds", str(STAGE_TIMEOUT_SECONDS),
    "--minimum-points", str(MINIMUM_POINTS),
    "--minimum-tetrahedra", str(MINIMUM_TETRAHEDRA),
    "--minimum-shape-quality", str(MINIMUM_TETRA_SHAPE_QUALITY),
]
if TIMEFRAME is not None:
    cmd.extend(["--timeframe", str(TIMEFRAME)])
if USE_GPU:
    cmd.append("--gpu")
if needs_opencarp:
    cmd.extend(["--carp-bin-dir", str(OPENCARP_BIN_DIR), "--allow-restricted-runtime"])

LAUNCHER_LOG = Path("/content") / f"cardianatomy-launcher-{RUN_ID}.log"
print("Output will be persisted in:", OUTPUT_ROOT)
print("Command:", shlex.join(cmd))
print("Starting upstream stages.")

start = time.monotonic()
last_report = 0.0
with LAUNCHER_LOG.open("w", encoding="utf-8") as launch_log:
    process = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=launch_log, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
    while process.poll() is None:
        time.sleep(20)
        elapsed = int(time.monotonic() - start)
        if time.monotonic() - last_report >= 60:
            last_report = time.monotonic()
            manifest_path = OUTPUT_ROOT / "run-manifest.json"
            if manifest_path.is_file():
                try:
                    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
                    completed = [stage.get("name") for stage in manifest.get("stages", []) if stage.get("status") == "completed"]
                    print(f"[{elapsed//60} min] status={manifest.get('status')} current_stage={manifest.get('current_stage')} completed={completed}")
                except Exception as exc:
                    print(f"[{elapsed//60} min] Manifest updating ({type(exc).__name__}).")
            else:
                print(f"[{elapsed//60} min] Waiting for initial manifest.")
    PIPELINE_EXIT_CODE = process.returncode

launcher_archive = DRIVE_RUNS_DIR / f"{CASE_ID}_{RUN_ID}_launcher.log"
shutil.copy2(LAUNCHER_LOG, launcher_archive)
print("Exit code:", PIPELINE_EXIT_CODE)
print("Launcher log copied to:", launcher_archive)
print("Launcher log tail:")
print("\n".join(LAUNCHER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-100:]))
print("Elapsed minutes:", round((time.monotonic() - start) / 60, 2))
if PIPELINE_EXIT_CODE != 0:
    print("Pipeline did not complete successfully. Keep OUTPUT_ROOT and inspect the manifest/logs before retrying.")

## 10. Inspect the manifest and mesh QC

This summary reports the recorded execution only. It does not validate segmentation labels, registration, cardiac anatomy, UVC fields, fibers or clinical performance.

In [ ]:
from pathlib import Path
import json

manifest_path = OUTPUT_ROOT / "run-manifest.json"
if not manifest_path.is_file():
    print("No run manifest was created. Keep the launcher log and inspect installation/preflight errors.")
else:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("Overall status:", manifest.get("status"))
    print("Upstream commit:", manifest.get("upstream", {}).get("commit_sha"))
    print("Selected stages:", manifest.get("selected_stages"))
    print("Checks:")
    print(json.dumps(manifest.get("checks", {}), indent=2))
    print("\nStage records:")
    for stage in manifest.get("stages", []):
        print("-", stage.get("name"), "|", stage.get("status"), "|", stage.get("error", ""))
    accepted = manifest.get("geometry_qc_passed_volume_mesh")
    if accepted:
        print("\nQC-passed volume mesh candidate:")
        print(json.dumps({
            "path": accepted.get("path"),
            "sha256": accepted.get("sha256"),
            "point_count": accepted.get("point_count"),
            "tetra_count": accepted.get("tetra_count"),
            "qc": accepted.get("qc"),
        }, indent=2))
    else:
        candidates = manifest.get("volume_mesh_candidates", [])
        print("\nNo QC-passed volume mesh was recorded. Candidate count:", len(candidates))
        for candidate in candidates[:10]:
            print(candidate.get("path"), "points=", candidate.get("point_count"),
                  "tetra=", candidate.get("tetra_count"),
                  "qc_passed=", candidate.get("qc", {}).get("passed"))
    print("\nPersisted outputs:", OUTPUT_ROOT)
    print("Review outputs against source images before scientific use.")

## 11. What to retain and what this does not prove

Retain the complete run folder in Drive, especially preflight-manifest.json, run-manifest.json, stage logs, input hashes, the upstream commit SHA, the produced geometry and its QC record.

Do not delete a failed run until its partial outputs have been inspected. For repeat attempts, rerun the configuration cell to generate a new timestamped output folder.

A successful command establishes only that the selected software produced recorded artifacts and that the adapter's configured QC checks passed. Independently review image/label overlays, cross-view alignment, segmentation quality, chamber boundaries, mesh topology, coordinate conventions and suitability for any downstream solver. This is a research workflow, not a clinical tool.